# Corte 2 · ETL y carga del Data Warehouse SITP Zonal

Pipeline del diseño del Corte 1: **Extract** (descarga de los zips diarios) → **Staging** (zips originales + bloques en memoria con pandas) → **Transform** (limpieza, conformación, claves subrogadas) → **Load** (modelo estrella en SQLite local).

Cadena de conexión local ): `sqlite:///dw_sitp.db`


In [5]:
!pip -q install holidays

## 1. Parámetros

In [6]:
import os, re, time, zipfile, sqlite3, codecs, shutil
from datetime import date, timedelta, datetime
import numpy as np
import pandas as pd
import requests

try:
    display
except NameError:
    display = print

# PARÁMETROS
FECHA_INICIO = date(2026, 9, 21)
FECHA_FIN    = date(2026, 10, 4)

BASE_URL     = "https://storage.googleapis.com/validaciones_tmsa/ValidacionZonal"

DIR_TRABAJO  = "/content/sitp"
DIR_STAGING  = os.path.join(DIR_TRABAJO, "staging")        # zona de staging: zips originales sin tocar
DB_PATH      = os.path.join(DIR_TRABAJO, "dw_sitp.db")     # bodega local

CHUNK        = 250_000      # filas por bloque en memoria
TOLERANCIA_SALDO = 1.0      # pesos de diferencia tolerados en Saldo_Previo - Valor = Saldo_Despues
RECHAZAR_SALDO_INCOHERENTE = False   # False: se carga y se marca con flag. True: se rechaza.

os.makedirs(DIR_STAGING, exist_ok=True)
DIAS = [FECHA_INICIO + timedelta(days=i) for i in range((FECHA_FIN - FECHA_INICIO).days + 1)]
print(f"{len(DIAS)} días: {DIAS[0]} -> {DIAS[-1]}")

14 días: 2026-09-21 -> 2026-10-04


## 2. Extract: descarga de los archivos diarios

Patrón del portal: `validacionZonalAAAAMMDD.zip`.

In [7]:
def url_dia(d):
    return f"{BASE_URL}/validacionZonal{d:%Y%m%d}.zip"

def ruta_zip(d):
    return os.path.join(DIR_STAGING, f"validacionZonal{d:%Y%m%d}.zip")

def descargar(d, intentos=3):
    destino = ruta_zip(d)
    if os.path.exists(destino) and zipfile.is_zipfile(destino):
        return destino
    for i in range(1, intentos + 1):
        try:
            with requests.get(url_dia(d), stream=True, timeout=120) as r:
                r.raise_for_status()
                tmp = destino + ".part"
                with open(tmp, "wb") as f:
                    for bloque in r.iter_content(1 << 20):
                        f.write(bloque)
            os.replace(tmp, destino)
            if not zipfile.is_zipfile(destino):
                raise IOError("el archivo descargado no es un zip válido")
            return destino
        except Exception as e:
            print(f"  intento {i}/{intentos} falló: {e}")
            time.sleep(5 * i)
    raise RuntimeError(f"No se pudo descargar {url_dia(d)}")

for d in DIAS:
    t0 = time.time()
    p = descargar(d)
    print(d, f"{os.path.getsize(p) / 1e6:6.1f} MB", f"{time.time() - t0:5.0f} s")

2026-09-21  127.0 MB     1 s
2026-09-22  131.1 MB     1 s
2026-09-23  130.4 MB     2 s
2026-09-24  129.9 MB     2 s
2026-09-25  128.4 MB     1 s
2026-09-26   87.5 MB     1 s
2026-09-27   49.3 MB     1 s
2026-09-28  127.9 MB     2 s
2026-09-29  130.5 MB     3 s
2026-09-30  130.7 MB     2 s
2026-10-01  130.1 MB     2 s
2026-10-02  130.0 MB     1 s
2026-10-03   89.1 MB     1 s
2026-10-04   51.7 MB     1 s


## 3. Perfilado rápido del primer archivo



In [8]:
def nombre_csv(zf):
    nombres = [n for n in zf.namelist() if n.lower().endswith(".csv")]
    if len(nombres) != 1:
        raise ValueError(f"Se esperaba 1 CSV dentro del zip y hay {len(nombres)}: {zf.namelist()}")
    return nombres[0]

def detectar_encoding(zf, nombre):
    with zf.open(nombre) as f:
        muestra = f.read(8_000_000)
    for enc in ("utf-8-sig", "latin-1"):
        try:
            codecs.getincrementaldecoder(enc)().decode(muestra)   # tolera un carácter cortado al final
            return enc
        except UnicodeDecodeError:
            continue
    return "latin-1"

with zipfile.ZipFile(ruta_zip(DIAS[0])) as zf:
    nom = nombre_csv(zf)
    enc = detectar_encoding(zf, nom)
    with zf.open(nom) as f:
        muestra = pd.read_csv(f, dtype=str, nrows=200_000, encoding=enc)

print(nom, "| encoding:", enc, "| forma de la muestra:", muestra.shape)
display(muestra.head(3))
print("\nNulos por columna:")
display(muestra.isna().sum()[lambda s: s > 0])
print("\nTipo_Tarifa:")
display(muestra["Tipo_Tarifa"].value_counts(dropna=False))
print("\nValor por Tipo_Tarifa:")
display(pd.to_numeric(muestra["Valor"], errors="coerce").groupby(muestra["Tipo_Tarifa"]).describe())
print("\nPerfiles:")
display(muestra["Nombre_Perfil"].value_counts(dropna=False))
print("\nHora_Pico_SN por hora del día:")
display(pd.crosstab(pd.to_datetime(muestra["Fecha_Transaccion"], errors="coerce").dt.hour, muestra["Hora_Pico_SN"]))

20260921.csv | encoding: utf-8-sig | forma de la muestra: (200000, 20)


,Dispositivo,Emisor,Estacion_Parada,Fase,Fecha_Clearing,Fecha_Transaccion,Hora_Pico_SN,ID_Vehiculo,Linea,Nombre_Perfil,Numero_Tarjeta,Operador,Ruta,Saldo_Despues_Transaccion,Saldo_Previo_a_Transaccion,Sistema,Tipo_Tarifa,Tipo_Tarjeta,Tipo_Vehiculo,Valor
0,220006485,(3101000) Bogota Card(Citizen),(53247) 005A12_TM|005A12_Estación Molinos,Fase 3,2026-09-21,2026-09-21 03:56:58,Peak Time,937183,(10127) HA708,(001) Adulto,3998c96a994ccfadcbf72a9737e91ad995cf325a12d2f2...,(018) GRAN AMERICAS USME,(11829) HA708_V2,47600.0,51150.0,ZONAL,1,tullave Plus,(2) Urbano,3550.0
1,220007513,(3101000) Bogota Card(Citizen),(53362) 052A12_TM|052A12_Br. La Reforma,Fase 3,2026-09-21,2026-09-21 03:57:02,Peak Time,937197,(10104) HK707,(001) Adulto,ef0a628f78c87e0d447bb8c94b7aa05e017844a897bcfd...,(018) GRAN AMERICAS USME,(10224) K707_Ida_SN_V1,8450.0,12000.0,ZONAL,1,tullave Plus,(2) Urbano,3550.0
2,220006485,(3101000) Bogota Card(Citizen),(53247) 005A12_TM|005A12_Estación Molinos,Fase 3,2026-09-21,2026-09-21 03:57:02,Peak Time,937183,(10127) HA708,(001) Adulto,e11c68f75071387ab37182f07c9da8a7abd1880267eb47...,(018) GRAN AMERICAS USME,(11829) HA708_V2,0.0,2700.0,ZONAL,1,tullave Plus,(2) Urbano,3550.0



Nulos por columna:


,0



Tipo_Tarifa:


...

,count,mean,std,min,25%,50%,75%,max
Tipo_Tarifa,,,,,,,,
1,149350.0,3550.0,0.0,3550.0,3550.0,3550.0,3550.0,3550.0
2,44385.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,6265.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Perfiles:


,count
Nombre_Perfil,
(001) Adulto,122293
(002) Adulto Mayor,25489
(001) Anonymous,23071
(006) Apoyo Ciudadano,10087
(009) Apoyo Ciudadano Reexpedición,8600
(101) Adulto PV,6277
(005) Discapacidad,4183



Hora_Pico_SN por hora del día:


Hora_Pico_SN,Peak Time
Fecha_Transaccion,
0,13
2,17
3,472
4,5862
5,15191
6,17052
7,13871
8,11012
9,9298


## 4. Esquema físico del Data Warehouse


Como es una carga inicial grande, la tabla de hechos se crea **sin índices** y estos se construyen al final (patrón *Bulk Insert* de la Clase 9). Las claves naturales de las dimensiones sí llevan `UNIQUE`.

In [9]:
DDL = '''
CREATE TABLE IF NOT EXISTS dim_tiempo (
    tiempo_sk INTEGER PRIMARY KEY AUTOINCREMENT,
    fecha TEXT NOT NULL, hora INTEGER NOT NULL,
    anio INTEGER, mes INTEGER, dia INTEGER, dia_semana INTEGER, nombre_dia TEXT,
    es_fin_de_semana INTEGER, es_festivo INTEGER, franja TEXT, hora_pico_fuente TEXT,
    UNIQUE (fecha, hora)
);
CREATE TABLE IF NOT EXISTS dim_estacion (
    estacion_sk INTEGER PRIMARY KEY AUTOINCREMENT,
    estacion_cod TEXT NOT NULL UNIQUE, estacion_nombre TEXT, fase TEXT
);
CREATE TABLE IF NOT EXISTS dim_ruta (
    ruta_sk INTEGER PRIMARY KEY AUTOINCREMENT,
    ruta_cod TEXT NOT NULL, linea_cod TEXT NOT NULL,
    ruta_nombre TEXT, linea_nombre TEXT, sistema TEXT,
    UNIQUE (ruta_cod, linea_cod)
);
CREATE TABLE IF NOT EXISTS dim_operador (
    operador_sk INTEGER PRIMARY KEY AUTOINCREMENT,
    operador_cod TEXT NOT NULL UNIQUE, operador_nombre TEXT
);
CREATE TABLE IF NOT EXISTS dim_perfil (
    perfil_sk INTEGER PRIMARY KEY AUTOINCREMENT,
    perfil_cod TEXT NOT NULL, perfil_nombre TEXT NOT NULL, tipo_tarjeta TEXT NOT NULL,
    es_subsidiado INTEGER,
    UNIQUE (perfil_cod, perfil_nombre, tipo_tarjeta)
);
CREATE TABLE IF NOT EXISTS dim_vehiculo (
    vehiculo_sk INTEGER PRIMARY KEY AUTOINCREMENT,
    id_vehiculo TEXT NOT NULL UNIQUE, tipo_vehiculo_cod TEXT, tipo_vehiculo_nombre TEXT
);
CREATE TABLE IF NOT EXISTS dim_tarifa (
    tarifa_sk INTEGER PRIMARY KEY AUTOINCREMENT,
    tipo_tarifa TEXT NOT NULL UNIQUE
);
CREATE TABLE IF NOT EXISTS fact_validaciones (
    tiempo_sk INTEGER NOT NULL REFERENCES dim_tiempo(tiempo_sk),
    estacion_sk INTEGER NOT NULL REFERENCES dim_estacion(estacion_sk),
    ruta_sk INTEGER NOT NULL REFERENCES dim_ruta(ruta_sk),
    operador_sk INTEGER NOT NULL REFERENCES dim_operador(operador_sk),
    perfil_sk INTEGER NOT NULL REFERENCES dim_perfil(perfil_sk),
    vehiculo_sk INTEGER NOT NULL REFERENCES dim_vehiculo(vehiculo_sk),
    tarifa_sk INTEGER NOT NULL REFERENCES dim_tarifa(tarifa_sk),
    numero_tarjeta TEXT,            -- dimensión degenerada
    dispositivo TEXT,               -- dimensión degenerada
    fecha_hora_transaccion TEXT,
    valor REAL,                     -- aditiva
    saldo_previo REAL,              -- semi-aditiva
    saldo_posterior REAL,           -- semi-aditiva
    flag_saldo_incoherente INTEGER, -- 1 si saldo_previo - valor != saldo_posterior
    archivo_fecha TEXT              -- linaje: de qué archivo diario viene
);
CREATE TABLE IF NOT EXISTS rechazados (
    Dispositivo TEXT, Emisor TEXT, Estacion_Parada TEXT, Fase TEXT, Fecha_Clearing TEXT,
    Fecha_Transaccion TEXT, Hora_Pico_SN TEXT, ID_Vehiculo TEXT, Linea TEXT, Nombre_Perfil TEXT,
    Numero_Tarjeta TEXT, Operador TEXT, Ruta TEXT, Saldo_Despues_Transaccion TEXT,
    Saldo_Previo_a_Transaccion TEXT, Sistema TEXT, Tipo_Tarifa TEXT, Tipo_Tarjeta TEXT,
    Tipo_Vehiculo TEXT, Valor TEXT,
    motivo TEXT, archivo_fecha TEXT
);
CREATE TABLE IF NOT EXISTS etl_control (
    archivo TEXT PRIMARY KEY, fecha_archivo TEXT, estado TEXT,
    filas_leidas INTEGER, filas_cargadas INTEGER, filas_rechazadas INTEGER, filas_duplicadas INTEGER,
    saldo_incoherente INTEGER, saldo_negativo INTEGER, fuera_de_fecha INTEGER,
    inicio TEXT, fin TEXT, mensaje TEXT
);
'''

con = sqlite3.connect(DB_PATH)
# Ajustes de velocidad para carga masiva. Si el proceso se corta de golpe, borra el .db y vuelve a correr.
con.execute("PRAGMA journal_mode = MEMORY")
con.execute("PRAGMA synchronous = OFF")
con.execute("PRAGMA temp_store = MEMORY")
con.execute("PRAGMA cache_size = -200000")
con.executescript(DDL)
con.commit()
print("Tablas:", [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")])

Tablas: ['dim_tiempo', 'dim_estacion', 'dim_ruta', 'dim_operador', 'dim_perfil', 'dim_vehiculo', 'dim_tarifa', 'fact_validaciones', 'rechazados', 'etl_control']


## 5. Transform

Reglas, en este orden de prioridad (un registro rechazado queda en la tabla `rechazados` con su motivo):

| Código | Regla | Acción |
|---|---|---|
| `R1_NULO_OBLIGATORIO` | Nulo en un campo obligatorio | Rechazar |
| `R2_FORMATO_INVALIDO` | Fecha, valor o saldos no interpretables | Rechazar |
| `R3_VALOR_NEGATIVO` | `Valor` < 0 | Rechazar |
| `R4_SALDO_INCOHERENTE` | `Saldo_Previo - Valor` ≠ `Saldo_Despues` | Marcar con flag (o rechazar si `RECHAZAR_SALDO_INCOHERENTE = True`) |
| duplicado | Misma tarjeta, dispositivo, vehículo, instante y valor dentro del bloque | Descartar y contar |

Conformación: los campos tipo `"(10330) LK810"` se parten en código y nombre; el nombre de la estación se limpia (`"089A13_TM|089A13_Br. La Gloria Oriental"` → `"Br. La Gloria Oriental"`); `es_subsidiado` se deriva del código de perfil (002, 005, 006 y 009).

In [10]:
COLS_ORIGEN = ["Dispositivo", "Emisor", "Estacion_Parada", "Fase", "Fecha_Clearing", "Fecha_Transaccion",
               "Hora_Pico_SN", "ID_Vehiculo", "Linea", "Nombre_Perfil", "Numero_Tarjeta", "Operador", "Ruta",
               "Saldo_Despues_Transaccion", "Saldo_Previo_a_Transaccion", "Sistema", "Tipo_Tarifa",
               "Tipo_Tarjeta", "Tipo_Vehiculo", "Valor"]
OBLIGATORIAS = ["Dispositivo", "Estacion_Parada", "Fecha_Transaccion", "ID_Vehiculo", "Linea", "Nombre_Perfil",
                "Numero_Tarjeta", "Operador", "Ruta", "Tipo_Tarifa", "Tipo_Tarjeta", "Valor",
                "Saldo_Previo_a_Transaccion", "Saldo_Despues_Transaccion"]
PERFILES_SUBSIDIADOS = {"002", "005", "006", "009"}
NOMBRES_DIA = {1: "Lunes", 2: "Martes", 3: "Miércoles", 4: "Jueves", 5: "Viernes", 6: "Sábado", 7: "Domingo"}

try:
    import holidays
    FESTIVOS = holidays.country_holidays("CO", years=range(DIAS[0].year - 1, DIAS[-1].year + 2))
except Exception as e:
    FESTIVOS = {}
    print("Aviso: sin librería 'holidays', es_festivo quedará en 0 ->", e)

def extraer_codigo(serie):
    # "(10330) LK810" -> ("10330", "LK810"). Si no trae paréntesis, el código es SIN_CODIGO.
    p = serie.str.extract(r"^\(\s*([^)]*?)\s*\)\s*(.*)$")
    cod = p[0].fillna("SIN_CODIGO")
    txt = p[1].where(p[0].notna(), serie).fillna("SIN_DATO").str.strip()
    return cod, txt

def franja_horaria(hora):
    return pd.Series(np.select(
        [hora <= 4, hora <= 8, hora <= 11, hora <= 15, hora <= 19],
        ["Madrugada (0-4)", "Mañana (5-8)", "Media mañana (9-11)", "Mediodía-tarde (12-15)", "Tarde-noche (16-19)"],
        default="Noche (20-23)"), index=hora.index)

def transformar(raw, d):
    # Devuelve (hechos_listos, rechazados, estadisticas) para un bloque de filas crudas.
    df = raw[COLS_ORIGEN].copy()
    for c in COLS_ORIGEN:
        s = df[c]
        df[c] = s.where(s.isna(), s.astype(str).str.strip()).replace("", np.nan)

    ts    = pd.to_datetime(df["Fecha_Transaccion"], errors="coerce")
    valor = pd.to_numeric(df["Valor"], errors="coerce")
    sprev = pd.to_numeric(df["Saldo_Previo_a_Transaccion"], errors="coerce")
    spost = pd.to_numeric(df["Saldo_Despues_Transaccion"], errors="coerce")

    r1 = df[OBLIGATORIAS].isna().any(axis=1)
    r2 = ts.isna() | valor.isna() | sprev.isna() | spost.isna()
    r3 = valor < 0
    incoherente = (sprev - valor - spost).abs() > TOLERANCIA_SALDO
    r4 = incoherente & RECHAZAR_SALDO_INCOHERENTE

    motivo = pd.Series(np.select(
        [r1, r2, r3, r4],
        ["R1_NULO_OBLIGATORIO", "R2_FORMATO_INVALIDO", "R3_VALOR_NEGATIVO", "R4_SALDO_INCOHERENTE"],
        default=""), index=df.index)

    rech = raw.loc[motivo != "", COLS_ORIGEN].copy()
    rech["motivo"] = motivo[motivo != ""]
    rech["archivo_fecha"] = d.isoformat()

    ok = df.loc[motivo == ""]
    dup = ok.duplicated(subset=["Dispositivo", "Numero_Tarjeta", "Fecha_Transaccion", "ID_Vehiculo", "Valor"], keep="first")
    n_dup = int(dup.sum())
    ok = ok.loc[~dup]
    ix = ok.index

    h = pd.DataFrame(index=ix)
    h["fecha"] = ts[ix].dt.strftime("%Y-%m-%d")
    h["hora"] = ts[ix].dt.hour.astype("int64")
    h["fecha_hora_transaccion"] = ts[ix].dt.strftime("%Y-%m-%d %H:%M:%S")
    h["hora_pico_fuente"] = ok["Hora_Pico_SN"].fillna("SIN_DATO")
    h["estacion_cod"], est_txt = extraer_codigo(ok["Estacion_Parada"])
    h["estacion_nombre"] = (est_txt.str.split("|").str[-1]
                            .str.replace(r"^[A-Za-z0-9]+_", "", regex=True).str.strip())
    h["fase"] = ok["Fase"].fillna("SIN_DATO")
    h["ruta_cod"], h["ruta_nombre"] = extraer_codigo(ok["Ruta"])
    h["linea_cod"], h["linea_nombre"] = extraer_codigo(ok["Linea"])
    h["sistema"] = ok["Sistema"].fillna("SIN_DATO")
    h["operador_cod"], h["operador_nombre"] = extraer_codigo(ok["Operador"])
    h["perfil_cod"], h["perfil_nombre"] = extraer_codigo(ok["Nombre_Perfil"])
    h["tipo_tarjeta"] = ok["Tipo_Tarjeta"]
    h["es_subsidiado"] = h["perfil_cod"].isin(PERFILES_SUBSIDIADOS).astype("int64")
    h["id_vehiculo"] = ok["ID_Vehiculo"]
    h["tipo_vehiculo_cod"], h["tipo_vehiculo_nombre"] = extraer_codigo(ok["Tipo_Vehiculo"])
    h["tipo_tarifa"] = ok["Tipo_Tarifa"]
    h["numero_tarjeta"] = ok["Numero_Tarjeta"]
    h["dispositivo"] = ok["Dispositivo"]
    h["valor"] = valor[ix]
    h["saldo_previo"] = sprev[ix]
    h["saldo_posterior"] = spost[ix]
    h["flag_saldo_incoherente"] = incoherente[ix].astype("int64")

    est = {
        "leidas": len(raw),
        "rechazadas": int((motivo != "").sum()),
        "duplicadas": n_dup,
        "cargadas": len(h),
        "saldo_incoherente": int(h["flag_saldo_incoherente"].sum()),
        "saldo_negativo": int(((h["saldo_previo"] < 0) | (h["saldo_posterior"] < 0)).sum()),
        "fuera_de_fecha": int((h["fecha"] != d.isoformat()).sum()),
        "motivos": motivo[motivo != ""].value_counts().to_dict(),
    }
    return h, rech, est

def dims_del_chunk(h):
    # Arma, para cada dimensión, las filas candidatas (claves naturales + atributos) de este bloque.
    t = h[["fecha", "hora", "hora_pico_fuente"]].drop_duplicates(["fecha", "hora"]).copy()
    f = pd.to_datetime(t["fecha"])
    t["anio"] = f.dt.year.astype("int64")
    t["mes"] = f.dt.month.astype("int64")
    t["dia"] = f.dt.day.astype("int64")
    t["dia_semana"] = (f.dt.dayofweek + 1).astype("int64")
    t["nombre_dia"] = t["dia_semana"].map(NOMBRES_DIA)
    t["es_fin_de_semana"] = (t["dia_semana"] >= 6).astype("int64")
    t["es_festivo"] = f.dt.date.map(lambda x: int(x in FESTIVOS)).astype("int64")
    t["franja"] = franja_horaria(t["hora"])
    return {
        "dim_tiempo": t,
        "dim_estacion": h[["estacion_cod", "estacion_nombre", "fase"]],
        "dim_ruta": h[["ruta_cod", "linea_cod", "ruta_nombre", "linea_nombre", "sistema"]],
        "dim_operador": h[["operador_cod", "operador_nombre"]],
        "dim_perfil": h[["perfil_cod", "perfil_nombre", "tipo_tarjeta", "es_subsidiado"]],
        "dim_vehiculo": h[["id_vehiculo", "tipo_vehiculo_cod", "tipo_vehiculo_nombre"]],
        "dim_tarifa": h[["tipo_tarifa"]],
    }

## 6. Load

Por cada dimensión: **lookup** por clave natural → si no existe, **insert** y se genera la SK → si existe y cambió un atributo, **SCD tipo 1** (sobrescribir). `dim_tiempo` y `dim_tarifa` son estáticas (solo insert). Después se traducen las claves naturales del hecho a SK.

In [11]:
DIMS = {
    "dim_tiempo":   dict(nat=["fecha", "hora"], attrs=["anio", "mes", "dia", "dia_semana", "nombre_dia", "es_fin_de_semana", "es_festivo", "franja", "hora_pico_fuente"], scd1=[]),
    "dim_estacion": dict(nat=["estacion_cod"], attrs=["estacion_nombre", "fase"], scd1=["estacion_nombre", "fase"]),
    "dim_ruta":     dict(nat=["ruta_cod", "linea_cod"], attrs=["ruta_nombre", "linea_nombre", "sistema"], scd1=["ruta_nombre", "linea_nombre", "sistema"]),
    "dim_operador": dict(nat=["operador_cod"], attrs=["operador_nombre"], scd1=["operador_nombre"]),
    "dim_perfil":   dict(nat=["perfil_cod", "perfil_nombre", "tipo_tarjeta"], attrs=["es_subsidiado"], scd1=["es_subsidiado"]),
    "dim_vehiculo": dict(nat=["id_vehiculo"], attrs=["tipo_vehiculo_cod", "tipo_vehiculo_nombre"], scd1=["tipo_vehiculo_cod", "tipo_vehiculo_nombre"]),
    "dim_tarifa":   dict(nat=["tipo_tarifa"], attrs=[], scd1=[]),
}
SK = {t: t.replace("dim_", "") + "_sk" for t in DIMS}
COLS_ENTERAS = {"hora", "anio", "mes", "dia", "dia_semana", "es_fin_de_semana", "es_festivo", "es_subsidiado"} | set(SK.values())

def a_tuplas(df):
    # Tipos nativos de Python (sqlite3 no acepta numpy.int64) y NaN -> None.
    return df.astype(object).where(df.notna(), None).values.tolist()

def leer_cache(con, tabla):
    df = pd.read_sql(f"SELECT * FROM {tabla}", con)
    return df.astype({c: "int64" for c in df.columns if c in COLS_ENTERAS})

def resolver_dimension(con, tabla, frame, cache):
    cfg = DIMS[tabla]
    nat, attrs, scd1, sk = cfg["nat"], cfg["attrs"], cfg["scd1"], SK[tabla]
    cols = nat + attrs
    cand = frame[cols].drop_duplicates(subset=nat)
    m = cand.merge(cache[tabla], on=nat, how="left", suffixes=("", "_act"))

    nuevos = m[m[sk].isna()]
    existentes = m[m[sk].notna()]
    cambios = existentes.iloc[0:0]
    if scd1 and len(existentes):
        dif = np.zeros(len(existentes), dtype=bool)
        for c in scd1:
            dif |= (existentes[c].values != existentes[c + "_act"].values)
        cambios = existentes[dif]

    if len(nuevos):
        con.executemany(f"INSERT INTO {tabla} ({','.join(cols)}) VALUES ({','.join('?' * len(cols))})",
                        a_tuplas(nuevos[cols]))
    if len(cambios):
        sets = ",".join(f"{c}=?" for c in scd1)
        con.executemany(f"UPDATE {tabla} SET {sets} WHERE {sk}=?",
                        a_tuplas(cambios[scd1 + [sk]].astype({sk: "int64"})))
    if len(nuevos) or len(cambios):
        cache[tabla] = leer_cache(con, tabla)
    return len(nuevos), len(cambios)

COLS_HECHO = [SK[t] for t in DIMS] + ["numero_tarjeta", "dispositivo", "fecha_hora_transaccion", "valor",
                                       "saldo_previo", "saldo_posterior", "flag_saldo_incoherente", "archivo_fecha"]

def cargar_hechos(con, h, d, cache):
    f = h
    for tabla, cfg in DIMS.items():
        sk = SK[tabla]
        f = f.merge(cache[tabla][cfg["nat"] + [sk]], on=cfg["nat"], how="left", validate="many_to_one")
        if f[sk].isna().any():
            raise ValueError(f"Lookup sin resultado en {tabla}")
    f["archivo_fecha"] = d.isoformat()
    con.executemany(f"INSERT INTO fact_validaciones ({','.join(COLS_HECHO)}) VALUES ({','.join('?' * len(COLS_HECHO))})",
                    a_tuplas(f[COLS_HECHO]))
    return len(f)

## 7. Pipeline por día

Cada archivo se procesa por bloques, con control en `etl_control`: estado `EN_PROCESO` → `OK` o `ERROR`. Al final de cada día se **concilia**: leídas = cargadas + rechazadas + duplicadas. Si no cuadra, el día queda en `ERROR`.

In [12]:
def procesar_dia(con, d, cache):
    archivo = os.path.basename(ruta_zip(d))
    previo = con.execute("SELECT estado FROM etl_control WHERE archivo=?", (archivo,)).fetchone()
    if previo and previo[0] == "OK":
        print(f"{d}: ya procesado, se omite")
        return
    if previo:   # un intento anterior quedó a medias: se limpia ese día y se reprocesa completo
        con.execute("DELETE FROM fact_validaciones WHERE archivo_fecha=?", (d.isoformat(),))
        con.execute("DELETE FROM rechazados WHERE archivo_fecha=?", (d.isoformat(),))
    inicio = datetime.now().isoformat(timespec="seconds")
    con.execute("INSERT OR REPLACE INTO etl_control (archivo, fecha_archivo, estado, inicio) VALUES (?,?,?,?)",
                (archivo, d.isoformat(), "EN_PROCESO", inicio))
    con.commit()

    tot = dict(leidas=0, cargadas=0, rechazadas=0, duplicadas=0, saldo_incoherente=0, saldo_negativo=0, fuera_de_fecha=0)
    motivos = {}
    t0 = time.time()
    try:
        with zipfile.ZipFile(ruta_zip(d)) as zf:
            nom = nombre_csv(zf)
            enc = detectar_encoding(zf, nom)
            with zf.open(nom) as f:
                for raw in pd.read_csv(f, dtype=str, chunksize=CHUNK, encoding=enc):
                    raw.columns = [c.strip() for c in raw.columns]
                    faltan = set(COLS_ORIGEN) - set(raw.columns)
                    if faltan:
                        raise ValueError(f"Columnas faltantes en el archivo: {sorted(faltan)}")
                    h, rech, est = transformar(raw, d)
                    if len(rech):
                        rech.to_sql("rechazados", con, if_exists="append", index=False)
                    if len(h):
                        dims = dims_del_chunk(h)
                        for tabla in DIMS:
                            resolver_dimension(con, tabla, dims[tabla], cache)
                        cargar_hechos(con, h, d, cache)
                    con.commit()
                    for k in tot:
                        tot[k] += est[k]
                    for k, v in est["motivos"].items():
                        motivos[k] = motivos.get(k, 0) + v
        if tot["leidas"] != tot["cargadas"] + tot["rechazadas"] + tot["duplicadas"]:
            raise ValueError(f"No concilia: {tot}")
        con.execute('''UPDATE etl_control SET estado='OK', filas_leidas=?, filas_cargadas=?, filas_rechazadas=?,
                       filas_duplicadas=?, saldo_incoherente=?, saldo_negativo=?, fuera_de_fecha=?, fin=?, mensaje=?
                       WHERE archivo=?''',
                    (tot["leidas"], tot["cargadas"], tot["rechazadas"], tot["duplicadas"], tot["saldo_incoherente"],
                     tot["saldo_negativo"], tot["fuera_de_fecha"], datetime.now().isoformat(timespec="seconds"),
                     str(motivos), archivo))
        con.commit()
        print(f"{d} | leídas {tot['leidas']:>9,} | cargadas {tot['cargadas']:>9,} | rechazadas {tot['rechazadas']:>6,} "
              f"| duplicadas {tot['duplicadas']:>5,} | saldo incoh. {tot['saldo_incoherente']:>6,} | {time.time() - t0:5.0f} s")
    except Exception as e:
        con.rollback()
        con.execute("UPDATE etl_control SET estado='ERROR', fin=?, mensaje=? WHERE archivo=?",
                    (datetime.now().isoformat(timespec="seconds"), str(e)[:500], archivo))
        con.commit()
        print(f"{d} | ERROR: {e}")
        raise

cache = {t: leer_cache(con, t) for t in DIMS}
for d in DIAS:
    procesar_dia(con, d, cache)

2026-09-21 | leídas 1,782,454 | cargadas 1,782,454 | rechazadas      0 | duplicadas     0 | saldo incoh. 153,819 |   112 s
2026-09-22 | leídas 1,844,214 | cargadas 1,844,214 | rechazadas      0 | duplicadas     0 | saldo incoh. 153,617 |   112 s
2026-09-23 | leídas 1,831,723 | cargadas 1,831,723 | rechazadas      0 | duplicadas     0 | saldo incoh. 155,012 |   113 s
2026-09-24 | leídas 1,825,487 | cargadas 1,825,486 | rechazadas      1 | duplicadas     0 | saldo incoh. 159,387 |   110 s
2026-09-25 | leídas 1,808,261 | cargadas 1,808,261 | rechazadas      0 | duplicadas     0 | saldo incoh. 172,706 |   109 s
2026-09-26 | leídas 1,233,586 | cargadas 1,233,584 | rechazadas      0 | duplicadas     2 | saldo incoh. 132,312 |    80 s
2026-09-27 | leídas   700,477 | cargadas   700,477 | rechazadas      0 | duplicadas     0 | saldo incoh. 82,000 |    44 s
2026-09-28 | leídas 1,801,106 | cargadas 1,801,106 | rechazadas      0 | duplicadas     0 | saldo incoh. 170,189 |   111 s
2026-09-29 | leíd

## 8. Índices (después de la carga masiva)

In [13]:
t0 = time.time()
for sk in SK.values():
    con.execute(f"CREATE INDEX IF NOT EXISTS ix_fact_{sk} ON fact_validaciones({sk})")
con.execute("CREATE INDEX IF NOT EXISTS ix_fact_archivo ON fact_validaciones(archivo_fecha)")
con.commit()
print(f"Índices creados en {time.time() - t0:.0f} s")

Índices creados en 208 s


## 9. Controles de calidad y trazabilidad


In [14]:
print("Resumen por archivo:")
display(pd.read_sql('''SELECT fecha_archivo, estado, filas_leidas, filas_cargadas, filas_rechazadas, filas_duplicadas,
                       saldo_incoherente, saldo_negativo, fuera_de_fecha FROM etl_control ORDER BY fecha_archivo''', con))

print("\nMotivos de rechazo:")
display(pd.read_sql("SELECT motivo, COUNT(*) AS filas FROM rechazados GROUP BY motivo", con))

print("\nIntegridad referencial (debe ser 0 en todas):")
huerfanas = {}
for tabla, sk in SK.items():
    huerfanas[tabla] = con.execute(
        f"SELECT COUNT(*) FROM fact_validaciones f LEFT JOIN {tabla} d ON f.{sk}=d.{sk} WHERE d.{sk} IS NULL").fetchone()[0]
display(pd.Series(huerfanas, name="FK huérfanas"))

print("\nFilas por tabla:")
display(pd.Series({t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                   for t in list(DIMS) + ["fact_validaciones", "rechazados"]}, name="filas"))

Resumen por archivo:


,fecha_archivo,estado,filas_leidas,filas_cargadas,filas_rechazadas,filas_duplicadas,saldo_incoherente,saldo_negativo,fuera_de_fecha
0,2026-09-21,OK,1782454,1782454,0,0,153819,0,3915
1,2026-09-22,OK,1844214,1844214,0,0,153617,0,3853
2,2026-09-23,OK,1831723,1831723,0,0,155012,0,3918
3,2026-09-24,OK,1825487,1825486,1,0,159387,0,3980
4,2026-09-25,OK,1808261,1808261,0,0,172706,0,2899
5,2026-09-26,OK,1233586,1233584,0,2,132312,0,574
6,2026-09-27,OK,700477,700477,0,0,82000,0,3778
7,2026-09-28,OK,1801106,1801106,0,0,170189,0,3937
8,2026-09-29,OK,1836529,1836529,0,0,162513,0,3999
9,2026-09-30,OK,1844182,1844182,0,0,178413,0,3861



Motivos de rechazo:


,motivo,filas
0,R1_NULO_OBLIGATORIO,1



Integridad referencial (debe ser 0 en todas):


,FK huérfanas
dim_tiempo,0
dim_estacion,0
dim_ruta,0
dim_operador,0
dim_perfil,0
dim_vehiculo,0
dim_tarifa,0



Filas por tabla:


,filas
dim_tiempo,336
dim_estacion,7101
dim_ruta,730
dim_operador,20
dim_perfil,8
dim_vehiculo,7320
dim_tarifa,3
fact_validaciones,22175277
rechazados,1


In [15]:
# Ejemplo ROLAP: recaudo y validaciones por operador, separando perfiles subsidiados de tarifa plena.
display(pd.read_sql('''
    SELECT o.operador_nombre, p.es_subsidiado,
           COUNT(*)      AS validaciones,
           SUM(f.valor)  AS recaudo
    FROM fact_validaciones f
    JOIN dim_operador o ON f.operador_sk = o.operador_sk
    JOIN dim_perfil   p ON f.perfil_sk   = p.perfil_sk
    GROUP BY o.operador_nombre, p.es_subsidiado
    ORDER BY recaudo DESC
    LIMIT 20''', con))

,operador_nombre,es_subsidiado,validaciones,recaudo
0,CONSORCIO EXPRESS USAQUEN,0,2479363,6.463638e+09
1,ETIB,0,2264776,6.111928e+09
2,MASIVO CAPITAL KENNEDY,0,1707124,4.358129e+09
3,GMOVIL ENGATIVA,0,1502262,4.042971e+09
4,SUMA,0,1144435,2.974268e+09
5,EMASIVO 16,0,782734,2.174513e+09
6,CONSORCIO EXPRESS SAN CRISTOBAL,0,850132,2.161698e+09
7,ETIB,1,786629,1.999800e+09
8,CONSORCIO EXPRESS USAQUEN,1,733404,1.777350e+09
9,EMASIVO 10,0,628538,1.710252e+09


## 10. Exportación para Power BI



In [17]:
# Exporta tablas pequeñas (dimensiones + hechos agregados) para Power BI y las baja a tu PC.
import shutil
EXPORT_DIR = os.path.join(DIR_TRABAJO, "export_powerbi")
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
os.makedirs(EXPORT_DIR)

def exportar(nombre, sql):
    df = pd.read_sql(sql, con)
    df.to_csv(os.path.join(EXPORT_DIR, nombre + ".csv"), index=False, encoding="utf-8-sig")
    print(f"{nombre:<18} {len(df):>10,} filas")

for t in DIMS:
    exportar(t, f"SELECT * FROM {t}")

exportar("agg_hora_ruta", '''
    SELECT tiempo_sk, ruta_sk, operador_sk, perfil_sk, tarifa_sk,
           COUNT(*) AS validaciones,
           SUM(valor) AS recaudo,
           SUM(CASE WHEN valor > 0 THEN 1 ELSE 0 END) AS validaciones_con_cobro
    FROM fact_validaciones
    GROUP BY tiempo_sk, ruta_sk, operador_sk, perfil_sk, tarifa_sk''')

exportar("agg_estacion_dia", '''
    SELECT t.fecha, f.estacion_sk, f.perfil_sk, f.tarifa_sk,
           COUNT(*) AS validaciones,
           SUM(f.valor) AS recaudo
    FROM fact_validaciones f
    JOIN dim_tiempo t ON t.tiempo_sk = f.tiempo_sk
    GROUP BY t.fecha, f.estacion_sk, f.perfil_sk, f.tarifa_sk''')

zip_path = shutil.make_archive(os.path.join(DIR_TRABAJO, "export_powerbi"), "zip", EXPORT_DIR)
print(f"\nZIP listo: {os.path.getsize(zip_path) / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Fuera de Colab:", zip_path)

dim_tiempo                336 filas
dim_estacion            7,101 filas
dim_ruta                  730 filas
dim_operador               20 filas
dim_perfil                  8 filas
dim_vehiculo            7,320 filas
dim_tarifa                  3 filas
agg_hora_ruta       1,546,856 filas
agg_estacion_dia      948,127 filas

ZIP listo: 10.3 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Consultas OLAP (ROLAP sobre SQLite)

Consultas que responden las 3 preguntas de BI con roll-up, drill-down, slice, dice y pivot. SQLite no tiene ROLLUP ni CUBE, por eso los subtotales se emulan con UNION ALL

In [19]:
import shutil, zipfile
pd.set_option("display.width", 200, "display.max_columns", 30, "display.float_format", "{:,.1f}".format)

OUT_DIR = os.path.join(DIR_TRABAJO, "olap")
shutil.rmtree(OUT_DIR, ignore_errors=True); os.makedirs(OUT_DIR)
RESULTADOS, CONSULTAS = {}, []

# Hora pico propia (la del origen viene constante): derivada de la demanda observada
TIPO_HORA = "CASE WHEN t.hora IN (5,6,7,8) THEN 'Pico manana' WHEN t.hora IN (16,17,18,19) THEN 'Pico tarde' ELSE 'Valle' END"

def olap(clave, titulo, sql, filas=15):
    df = pd.read_sql(sql, con)
    RESULTADOS[clave] = df
    CONSULTAS.append(f"-- {clave}: {titulo}\n{sql.strip()};\n")
    df.to_csv(os.path.join(OUT_DIR, clave + ".csv"), index=False, encoding="utf-8-sig")
    print(f"\n### {clave} - {titulo}  ({len(df):,} filas)")
    print(df.head(filas).to_string(index=False))

#  PREGUNTA 1
# ROLL-UP: operador x franja -> subtotal por operador -> total general (ROLLUP emulado con UNION ALL)
olap("q1_rollup", "Roll-up demanda: operador x franja, subtotales y total", '''
SELECT * FROM (
SELECT o.operador_nombre AS operador, t.franja AS franja, COUNT(*) AS validaciones
FROM fact_validaciones f JOIN dim_tiempo t ON t.tiempo_sk=f.tiempo_sk
JOIN dim_operador o ON o.operador_sk=f.operador_sk
GROUP BY o.operador_nombre, t.franja
UNION ALL
SELECT o.operador_nombre, 'TOTAL OPERADOR', COUNT(*)
FROM fact_validaciones f JOIN dim_operador o ON o.operador_sk=f.operador_sk
GROUP BY o.operador_nombre
UNION ALL
SELECT 'TOTAL GENERAL', '', COUNT(*) FROM fact_validaciones
) ORDER BY (operador='TOTAL GENERAL'), operador, (franja LIKE 'TOTAL%'), franja''', filas=40)

# DRILL-DOWN: franja -> hora (de la franja con mas demanda)
olap("q1_drilldown", "Drill-down: de franja a hora del dia", '''
SELECT t.franja, t.hora, COUNT(*) AS validaciones,
       ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 2) AS pct_total
FROM fact_validaciones f JOIN dim_tiempo t ON t.tiempo_sk=f.tiempo_sk
GROUP BY t.franja, t.hora ORDER BY t.hora''', filas=30)

# SLICE: solo dias habiles (es_fin_de_semana = 0), por tipo de hora
olap("q1_slice", "Slice: solo dias habiles, pico vs valle", f'''
SELECT {TIPO_HORA} AS tipo_hora,
       COUNT(*) AS validaciones,
       COUNT(DISTINCT t.tiempo_sk) AS horas_observadas,
       ROUND(1.0*COUNT(*)/COUNT(DISTINCT t.tiempo_sk), 0) AS validaciones_por_hora
FROM fact_validaciones f JOIN dim_tiempo t ON t.tiempo_sk=f.tiempo_sk
WHERE t.es_fin_de_semana = 0
GROUP BY 1 ORDER BY validaciones_por_hora DESC''')

# DICE: 3 operadores con mas demanda x franjas pico -> top rutas
olap("q1_dice", "Dice: top operadores x franjas pico, 15 rutas con mas demanda", '''
SELECT r.linea_nombre AS linea, r.ruta_nombre AS ruta, o.operador_nombre AS operador, COUNT(*) AS validaciones
FROM fact_validaciones f
JOIN dim_tiempo t ON t.tiempo_sk=f.tiempo_sk
JOIN dim_ruta r ON r.ruta_sk=f.ruta_sk
JOIN dim_operador o ON o.operador_sk=f.operador_sk
WHERE t.franja IN ('Mañana (5-8)','Tarde-noche (16-19)')
  AND o.operador_sk IN (SELECT operador_sk FROM fact_validaciones GROUP BY operador_sk ORDER BY COUNT(*) DESC LIMIT 3)
GROUP BY r.linea_nombre, r.ruta_nombre, o.operador_nombre
ORDER BY validaciones DESC LIMIT 15''')

# PIVOT: dia de la semana (filas) x franja (columnas)
olap("q1_pivot", "Pivot: dia de la semana x franja", '''
SELECT t.nombre_dia AS dia,
  SUM(CASE WHEN t.franja LIKE 'Madrugada%'  THEN 1 ELSE 0 END) AS madrugada,
  SUM(CASE WHEN t.franja LIKE 'Mañana%'     THEN 1 ELSE 0 END) AS manana,
  SUM(CASE WHEN t.franja LIKE 'Media%'      THEN 1 ELSE 0 END) AS media_manana,
  SUM(CASE WHEN t.franja LIKE 'Mediodía%'   THEN 1 ELSE 0 END) AS mediodia_tarde,
  SUM(CASE WHEN t.franja LIKE 'Tarde%'      THEN 1 ELSE 0 END) AS tarde_noche,
  SUM(CASE WHEN t.franja LIKE 'Noche%'      THEN 1 ELSE 0 END) AS noche,
  COUNT(*) AS total
FROM fact_validaciones f JOIN dim_tiempo t ON t.tiempo_sk=f.tiempo_sk
GROUP BY t.dia_semana, t.nombre_dia ORDER BY t.dia_semana''')

#  PREGUNTA 2
# ROLL-UP: recaudo por operador y tipo de perfil (subsidiado / tarifa plena), con subtotal y total
olap("q2_rollup", "Roll-up recaudo: operador x tipo de perfil, subtotales y total", '''
SELECT * FROM (
SELECT o.operador_nombre AS operador,
       CASE p.es_subsidiado WHEN 1 THEN 'Subsidiado' ELSE 'Tarifa plena' END AS tipo_perfil,
       COUNT(*) AS validaciones, SUM(f.valor) AS recaudo_nominal
FROM fact_validaciones f JOIN dim_operador o ON o.operador_sk=f.operador_sk
JOIN dim_perfil p ON p.perfil_sk=f.perfil_sk
GROUP BY o.operador_nombre, p.es_subsidiado
UNION ALL
SELECT o.operador_nombre, 'TOTAL OPERADOR', COUNT(*), SUM(f.valor)
FROM fact_validaciones f JOIN dim_operador o ON o.operador_sk=f.operador_sk
GROUP BY o.operador_nombre
UNION ALL
SELECT 'TOTAL GENERAL', '', COUNT(*), SUM(valor) FROM fact_validaciones
) ORDER BY (operador='TOTAL GENERAL'), operador, (tipo_perfil LIKE 'TOTAL%'), tipo_perfil''', filas=60)

# DRILL-DOWN: operador -> ruta (recaudo y % subsidiado por ruta)
olap("q2_drilldown", "Drill-down: de operador a ruta (recaudo y % subsidiado)", '''
SELECT o.operador_nombre AS operador, r.ruta_nombre AS ruta,
       SUM(f.valor) AS recaudo_nominal,
       ROUND(100.0*SUM(CASE WHEN p.es_subsidiado=1 THEN f.valor ELSE 0 END)/NULLIF(SUM(f.valor),0), 1) AS pct_recaudo_subsidiado,
       ROUND(100.0*SUM(CASE WHEN p.es_subsidiado=1 THEN 1 ELSE 0 END)/COUNT(*), 1) AS pct_validaciones_subsidiadas
FROM fact_validaciones f
JOIN dim_operador o ON o.operador_sk=f.operador_sk
JOIN dim_ruta r ON r.ruta_sk=f.ruta_sk
JOIN dim_perfil p ON p.perfil_sk=f.perfil_sk
GROUP BY o.operador_nombre, r.ruta_nombre
ORDER BY recaudo_nominal DESC LIMIT 20''', filas=20)

# SLICE: solo perfiles subsidiados, por perfil
olap("q2_slice", "Slice: solo perfiles subsidiados", '''
SELECT p.perfil_nombre AS perfil, COUNT(*) AS validaciones, SUM(f.valor) AS recaudo_nominal,
       SUM(CASE WHEN f.valor=0 THEN 1 ELSE 0 END) AS validaciones_sin_cobro
FROM fact_validaciones f JOIN dim_perfil p ON p.perfil_sk=f.perfil_sk
WHERE p.es_subsidiado = 1
GROUP BY p.perfil_nombre ORDER BY validaciones DESC''')

# PIVOT: operador (filas) x tipo de perfil (columnas) con proporcion
olap("q2_pivot", "Pivot: operador x (subsidiado / tarifa plena), recaudo y proporcion", '''
SELECT o.operador_nombre AS operador,
  SUM(CASE WHEN p.es_subsidiado=1 THEN f.valor ELSE 0 END) AS recaudo_subsidiado,
  SUM(CASE WHEN p.es_subsidiado=0 THEN f.valor ELSE 0 END) AS recaudo_tarifa_plena,
  SUM(f.valor) AS recaudo_total,
  ROUND(100.0*SUM(CASE WHEN p.es_subsidiado=1 THEN f.valor ELSE 0 END)/NULLIF(SUM(f.valor),0), 2) AS pct_subsidiado
FROM fact_validaciones f JOIN dim_operador o ON o.operador_sk=f.operador_sk
JOIN dim_perfil p ON p.perfil_sk=f.perfil_sk
GROUP BY o.operador_nombre ORDER BY recaudo_total DESC''', filas=25)

#  PREGUNTA 3
# ROLL-UP/DRILL: perfil x operador (zona) con participacion dentro de cada operador
olap("q3_perfil_zona", "Perfil x zona (operador): validaciones y % dentro de la zona", '''
WITH base AS (
  SELECT o.operador_nombre AS zona, p.perfil_nombre AS perfil, p.es_subsidiado AS sub, COUNT(*) AS validaciones
  FROM fact_validaciones f JOIN dim_operador o ON o.operador_sk=f.operador_sk
  JOIN dim_perfil p ON p.perfil_sk=f.perfil_sk
  GROUP BY o.operador_nombre, p.perfil_nombre, p.es_subsidiado)
SELECT zona, perfil, sub AS es_subsidiado, validaciones,
       ROUND(100.0*validaciones/SUM(validaciones) OVER (PARTITION BY zona), 2) AS pct_en_zona
FROM base ORDER BY zona, validaciones DESC''', filas=40)

# DICE: subsidiado vs tarifa plena en tres franjas
olap("q3_dice", "Dice: subsidiado vs tarifa plena en franjas clave", '''
SELECT t.franja, CASE p.es_subsidiado WHEN 1 THEN 'Subsidiado' ELSE 'Tarifa plena' END AS tipo_perfil,
       COUNT(*) AS validaciones,
       ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (PARTITION BY p.es_subsidiado), 2) AS pct_del_grupo
FROM fact_validaciones f JOIN dim_tiempo t ON t.tiempo_sk=f.tiempo_sk
JOIN dim_perfil p ON p.perfil_sk=f.perfil_sk
WHERE t.franja IN ('Mañana (5-8)','Mediodía-tarde (12-15)','Tarde-noche (16-19)')
GROUP BY t.franja, p.es_subsidiado ORDER BY t.franja, tipo_perfil''')

# PIVOT: perfil x tipo de dia (habil / fin de semana), patron de uso
olap("q3_pivot", "Pivot: perfil x tipo de dia (habil vs fin de semana)", '''
SELECT p.perfil_nombre AS perfil,
  SUM(CASE WHEN t.es_fin_de_semana=0 THEN 1 ELSE 0 END) AS dias_habiles,
  SUM(CASE WHEN t.es_fin_de_semana=1 THEN 1 ELSE 0 END) AS fin_de_semana,
  ROUND(100.0*SUM(CASE WHEN t.es_fin_de_semana=1 THEN 1 ELSE 0 END)/COUNT(*), 1) AS pct_fin_de_semana,
  ROUND(100.0*SUM(CASE WHEN f.valor=0 THEN 1 ELSE 0 END)/COUNT(*), 1) AS pct_sin_cobro
FROM fact_validaciones f JOIN dim_tiempo t ON t.tiempo_sk=f.tiempo_sk
JOIN dim_perfil p ON p.perfil_sk=f.perfil_sk
GROUP BY p.perfil_nombre ORDER BY (dias_habiles+fin_de_semana) DESC''')

#  empaquetar
with open(os.path.join(OUT_DIR, "consultas_olap.sql"), "w", encoding="utf-8") as fh:
    fh.write("\n".join(CONSULTAS))
zip_olap = shutil.make_archive(os.path.join(DIR_TRABAJO, "resultados_olap"), "zip", OUT_DIR)
print(f"\nListo: {len(RESULTADOS)} consultas. ZIP: {os.path.getsize(zip_olap)/1e3:.0f} KB")
try:
    from google.colab import files
    files.download(zip_olap)
except ImportError:
    print("Fuera de Colab:", zip_olap)


### q1_rollup - Roll-up demanda: operador x franja, subtotales y total  (141 filas)
                       operador                 franja  validaciones
CONSORCIO EXPRESS SAN CRISTOBAL        Madrugada (0-4)         46211
CONSORCIO EXPRESS SAN CRISTOBAL           Mañana (5-8)        345671
CONSORCIO EXPRESS SAN CRISTOBAL    Media mañana (9-11)        176552
CONSORCIO EXPRESS SAN CRISTOBAL Mediodía-tarde (12-15)        267071
CONSORCIO EXPRESS SAN CRISTOBAL          Noche (20-23)         75937
CONSORCIO EXPRESS SAN CRISTOBAL    Tarde-noche (16-19)        300693
CONSORCIO EXPRESS SAN CRISTOBAL         TOTAL OPERADOR       1212135
      CONSORCIO EXPRESS USAQUEN        Madrugada (0-4)         82278
      CONSORCIO EXPRESS USAQUEN           Mañana (5-8)        970207
      CONSORCIO EXPRESS USAQUEN    Media mañana (9-11)        483167
      CONSORCIO EXPRESS USAQUEN Mediodía-tarde (12-15)        711170
      CONSORCIO EXPRESS USAQUEN          Noche (20-23)        175823
      CONSORCIO EX

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>